In [1]:
import json
import chromadb
from chromadb.utils import embedding_functions
import os

In [2]:
# ── 1. Load scraped chunks ──────────────────────────────────────
with open("../genz_bible_data/genz_bible_chunks.json") as f:
    chunks = json.load(f)

print(f"✅ Loaded {len(chunks)} chunks")

✅ Loaded 1758 chunks


In [ ]:
def prepare_bible_documents(chunks_path: str) -> dict:
    """
    Prepare documents, metadatas, and ids from scraped JSON chunks.
    - document (indexed for search): KJV original text
    - genz_text (stored in metadata): GenZ translation used for answers
    """
    with open(chunks_path) as f:
        chunks = json.load(f)

    documents = []
    metadatas = []
    ids = []

    seen_ids = set()

    for c in chunks:
        # Skip malformed chunks (verse=0 until scraper is fixed)
        if c["verse"] == 0:
            continue

        # Deduplicate by id
        if c["id"] in seen_ids:
            continue
        seen_ids.add(c["id"])

        # Index KJV text for semantic search
        documents.append(c["text"])
        metadatas.append({
            "reference": c["reference"],
            "book": c["book"],
            "chapter": str(c["chapter"]),
            "verse": str(c["verse"]),
            "testament": c["testament"] or "",
            # GenZ translation stored in metadata — used for the agent's answer
            "genz_text": c.get("genz_text", ""),
        })
        ids.append(c["id"])

    print(f"Prepared {len(documents)} verse chunks")
    return {"documents": documents, "metadatas": metadatas, "ids": ids}

In [ ]:
def setup_bible_chromadb(
    chunks_path: str = "../genz_bible_data/genz_bible_chunks.json",
    collection_name: str = "genz_bible",
    chroma_path: str = "../chroma"
):
    """
    Create and populate ChromaDB collection with GenZ Bible data.
    """
    # ── Embedding model ─────────────────────────────────────────
    print("Loading embedding model (downloading if first time ~120MB)...")
    ef = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="sentence-transformers/all-MiniLM-L12-v2"
    )
    print("✅ Embedding model ready")

    # ── ChromaDB setup ───────────────────────────────────────────
    client = chromadb.PersistentClient(chroma_path)

    # Create collection (delete if exists)
    try:
        client.delete_collection(collection_name)
        print(f"🗑️  Deleted existing '{collection_name}' collection")
    except Exception:
        pass

    collection = client.create_collection(
        name=collection_name,
        embedding_function=ef,
        metadata={
            "description": "GenZ Bible translation for RAG-powered chatbot"
        }
    )

    # Prepare and load documents
    data = prepare_bible_documents(chunks_path)

    # Add to ChromaDB in batches (avoids memory issues for large datasets)
    batch_size = 500
    total_batches = (len(data["documents"]) + batch_size - 1) // batch_size
    for i in range(0, len(data["documents"]), batch_size):
        batch_num = i // batch_size + 1
        collection.add(
            documents=data["documents"][i:i + batch_size],
            metadatas=data["metadatas"][i:i + batch_size],
            ids=data["ids"][i:i + batch_size],
        )
        print(f"  ✅ Batch {batch_num}/{total_batches} loaded")

    print(f"\n🎉 Added {collection.count()} verses to ChromaDB collection '{collection_name}'")
    return collection, ef

In [6]:
# ── Run it ──────────────────────────────────────────────────────
collection, ef = setup_bible_chromadb()

# Sanity check queries
test_queries = [
    "Jesus turning water into wine",
    "For God so loved the world",
    "I am the light of the world",
]

for query in test_queries:
    print(f"\n🔍 Q: {query}")
    results = collection.query(query_texts=[query], n_results=2)
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        print(f"  → {meta['reference']} — {doc}")

Loading embedding model (downloading if first time ~120MB)...
✅ Embedding model ready
🗑️  Deleted existing 'genz_bible' collection
Prepared 21 verse chunks
  ✅ Batch 1/1 loaded

🎉 Added 21 verses to ChromaDB collection 'genz_bible'

🔍 Q: Jesus turning water into wine
  → John 6 — So, like, Jesus decided to take a trip across the sea of Galilee, you know, the one that's sometimes called the sea of Tiberias.
  → John 18 — After Jesus dropped these lines, he and his squad headed over to the Cedron stream. There was this lit garden on the other side, so they all rolled in. Yolo!

🔍 Q: For God so loved the world
  → John 13 — So, right before the feast of the passover, Jesus knew it was time to peace out of this world and go back to the Father . He had mad love for his squad that was chillin' in the world, and he loved them to the max.
  → John 1 — So like, when everything first started, there was the Word . And the Word was with God , and the Word was actually God !

🔍 Q: I am the light of